# Customer purchases — common data cleaning
This notebook prepares one cleaned CSV for every classifier. It performs schema checks and deterministic repairs only. Learned imputation, scaling and optional feature expansion belong inside the training/CV folds of each model notebook.

The source has no customer ID. RowId is the original one-based CSV row number and is kept for predictions, never for fitting. Repeated equivalent labeled rows are removed conservatively to avoid exact copies crossing a split; different purchase outcomes for the same age/salary profile are retained. This policy cannot establish whether identical source rows represent the same person.


## Setup and raw records


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split

locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations for candidate in (location, location / "classification")
     if (candidate / "customer_purchases_dataset/customer_details.csv").is_file()), None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root, classification/, or notebooks/.")
DATA_DIR = PROJECT_ROOT / "customer_purchases_dataset"
SOURCE_PATH = DATA_DIR / "customer_details.csv"
CLEANED_PATH = DATA_DIR / "cleaned_data.csv"
COMMON_ARTIFACTS = PROJECT_ROOT / "artifacts/common"
TARGET = "Purchased"
ID_COLUMN = "RowId"
FEATURE_COLUMNS = ["Age", "EstimatedSalary"]
CLASS_NAMES = ["Did not purchase", "Purchased"]
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
raw_data = pd.read_csv(SOURCE_PATH)
display(raw_data.head())
display(raw_data.isna().sum().to_frame("missing_count"))
print("Original rows:", len(raw_data))


,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0


,missing_count
Age,0
EstimatedSalary,0
Purchased,0


Original rows: 400


## Shared cleaning rules
1. Require Age, EstimatedSalary and Purchased; parse numeric values and mark nonfinite/malformed values missing.
2. Exclude records without a known binary label (0 or 1).
3. Mark nonpositive ages and negative estimated salaries missing; retain valid extremes.
4. Exclude records with both predictors missing, then drop repeated equivalent labeled records.
5. Keep remaining predictor gaps for model-specific, training-fitted median imputation.

No target-derived features, quantile clipping, learned scaling or global imputation are applied.


In [3]:
def clean_purchase_data(source):
    """Shared deterministic cleaning; no imputation or learned scaling here."""
    required = [*FEATURE_COLUMNS, TARGET]
    missing = set(required) - set(source.columns)
    if missing:
        raise ValueError(f"Missing required source columns: {sorted(missing)}")
    data = source[required].copy()
    data.insert(0, ID_COLUMN, np.arange(1, len(source) + 1))
    parsing_counts = {}
    for column in required:
        original = data[column].copy()
        data[column] = pd.to_numeric(data[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        parsing_counts[column] = int((original.notna() & data[column].isna()).sum())
    valid_label = data[TARGET].isin([0, 1])
    rejected_labels = data.loc[~valid_label].copy()
    rejected_labels["reason"] = "Missing, nonfinite or non-binary Purchased label"
    data = data.loc[valid_label].copy()
    data[FEATURE_COLUMNS] = data[FEATURE_COLUMNS].astype(float)
    invalid_age = data.Age.notna() & (data.Age <= 0)
    invalid_salary = data.EstimatedSalary.notna() & (data.EstimatedSalary < 0)
    data.loc[invalid_age, "Age"] = np.nan
    data.loc[invalid_salary, "EstimatedSalary"] = np.nan
    no_features = data[FEATURE_COLUMNS].isna().all(axis=1)
    rejected_features = data.loc[no_features].copy()
    rejected_features["reason"] = "Both predictors missing or invalid"
    data = data.loc[~no_features].copy()
    # There is no customer identifier in this source. Deduplicate equivalent
    # labeled records conservatively; retain different outcomes for a profile.
    duplicate = data.duplicated(subset=required, keep="first")
    duplicate_rows = data.loc[duplicate].copy()
    duplicate_rows["reason"] = "Repeated equivalent labeled record"
    data = data.loc[~duplicate].copy()
    data[TARGET] = data[TARGET].astype(int)
    data = data.reset_index(drop=True)
    if set(data[TARGET].unique()) != {0, 1} or data[TARGET].value_counts().min() < 4:
        raise ValueError("Need both classes with at least four usable rows per class")
    if data[FEATURE_COLUMNS].isna().all().any():
        raise ValueError("A predictor is entirely missing; cannot fit a training median")
    audit = {
        "source_rows": len(source), "invalid_label_rows_excluded": len(rejected_labels),
        "all_predictors_missing_rows_excluded": len(rejected_features),
        "duplicate_labeled_rows_excluded": int(duplicate.sum()), "cleaned_rows": len(data),
        "invalid_age_values_set_missing": int(invalid_age.sum()),
        "invalid_salary_values_set_missing": int(invalid_salary.sum()),
        "non_numeric_or_nonfinite_values": parsing_counts,
        "remaining_missing_predictors": data[FEATURE_COLUMNS].isna().sum().to_dict(),
        "class_counts": data[TARGET].value_counts().sort_index().to_dict(),
    }
    excluded = pd.concat([rejected_labels, rejected_features, duplicate_rows], ignore_index=True)
    return data, audit, excluded


In [4]:
cleaned_data, cleaning_audit, excluded_rows = clean_purchase_data(raw_data)
display(pd.Series(cleaning_audit, name="cleaning_audit").to_frame())
display(cleaned_data.head(10))
display(excluded_rows.head(20))
assert cleaned_data[ID_COLUMN].is_unique
assert set(cleaned_data[TARGET]) == {0, 1}
assert not cleaned_data.duplicated(subset=[*FEATURE_COLUMNS, TARGET]).any()
assert len(cleaned_data) + len(excluded_rows) == len(raw_data)


,cleaning_audit
source_rows,400
invalid_label_rows_excluded,0
all_predictors_missing_rows_excluded,0
duplicate_labeled_rows_excluded,33
cleaned_rows,367
invalid_age_values_set_missing,0
invalid_salary_values_set_missing,0
non_numeric_or_nonfinite_values,"{'Age': 0, 'EstimatedSalary': 0, 'Purchased': 0}"
remaining_missing_predictors,"{'Age': 0, 'EstimatedSalary': 0}"
class_counts,"{0: 229, 1: 138}"


,RowId,Age,EstimatedSalary,Purchased
0,1,19.0,19000.0,0
1,2,35.0,20000.0,0
2,3,26.0,43000.0,0
3,4,27.0,57000.0,0
4,5,19.0,76000.0,0
5,6,27.0,58000.0,0
6,7,27.0,84000.0,0
7,8,32.0,150000.0,1
8,9,25.0,33000.0,0
9,10,35.0,65000.0,0


,RowId,Age,EstimatedSalary,Purchased,reason
0,24,45.0,22000.0,1,Repeated equivalent labeled record
1,55,27.0,58000.0,0,Repeated equivalent labeled record
2,75,32.0,18000.0,0,Repeated equivalent labeled record
3,87,24.0,55000.0,0,Repeated equivalent labeled record
4,95,29.0,83000.0,0,Repeated equivalent labeled record
5,139,28.0,59000.0,0,Repeated equivalent labeled record
6,151,26.0,15000.0,0,Repeated equivalent labeled record
7,187,20.0,82000.0,0,Repeated equivalent labeled record
8,193,29.0,43000.0,0,Repeated equivalent labeled record
9,196,34.0,43000.0,0,Repeated equivalent labeled record


## Save common data and audit
Running this notebook overwrites the current shared files. Each classifier checks their source/cleaned-data fingerprints before fitting. After changing cleaning rules or the source, rerun this notebook and the classifier notebooks whose reports you want to refresh.


In [5]:
from datetime import datetime, timezone
COMMON_ARTIFACTS.mkdir(parents=True, exist_ok=True)
cleaned_data.to_csv(CLEANED_PATH, index=False)
excluded_rows.to_csv(COMMON_ARTIFACTS / "excluded_rows.csv", index=False)
pd.Series({key: value for key, value in cleaning_audit.items() if not isinstance(value, dict)}, name="value").to_csv(COMMON_ARTIFACTS / "data_cleaning_audit.csv")
cleaning_metadata = {
    "format_version": 1, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_sha256": hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest(),
    "cleaned_sha256": hashlib.sha256(CLEANED_PATH.read_bytes()).hexdigest(),
    "features": FEATURE_COLUMNS, "target": TARGET, "audit": cleaning_audit,
}
(COMMON_ARTIFACTS / "cleaning_metadata.json").write_text(json.dumps(cleaning_metadata, indent=2), encoding="utf-8")
print("Cleaned CSV:", CLEANED_PATH.relative_to(PROJECT_ROOT))
print("Usable rows:", len(cleaned_data))


Cleaned CSV: customer_purchases_dataset/cleaned_data.csv
Usable rows: 367


## Cleaning summary
Cleaning removes 33 repeated labeled records and retains 367 rows: 229 non-purchases and 138 purchases. No remaining predictor values are missing in this source. Identical profiles may belong to different people because there is no customer identifier; deduplication is a practical rule to keep exact copies out of different splits.

The cleaned CSV is shared by all seven classifiers. Each 03 notebook fits and reports its own model.
